# Stray Light Analysis

**Stray light** is any light that reaches the detector via an unintended path.
In imaging systems the most common form is **ghost images** — weak secondary
images caused by multiple internal reflections from lens surfaces.

The NSQ engine is ideal for stray light analysis because rays propagate freely
between any surfaces and every Fresnel reflection is tracked probabilistically.

This notebook demonstrates:
1. Converting a sequential design to an NSQ scene with `sequential_to_nonsequential`
2. Using `max_depth` to reveal ghost contributions
3. Comparing single-pass vs. multi-bounce irradiance

In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import warnings

from optiland.coordinate_system import CoordinateSystem
from optiland.nonsequential import (
    NSQScene, Spectrum,
    CollimatedSourceConfig,
    IrradianceDetectorConfig,
    LensConfig,
    sequential_to_nonsequential,
)

## 1. Why Sequential-to-NSQ Conversion?

The sequential tracer assumes all light is transmitted at every surface —
there are no reflected beams. The NSQ tracer applies **Fresnel splitting** at
every uncoated interface: each ray is probabilistically refracted *or* reflected
based on the Fresnel equations.

`sequential_to_nonsequential` converts an `Optic` sequential design into an
`NSQScene` automatically:
- Singlet surfaces → `Lens` components
- Cemented doublets → `Doublet` components  
- Mirror surfaces → `Mirror` components
- Image surface → `IrradianceDetector`
- Each sequential field → one NSQ source

In [2]:
from optiland.samples.objectives import CookeTriplet

triplet = CookeTriplet()

# Suppress the expected Fresnel-reflection warning from the converter
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    scene_triplet = sequential_to_nonsequential(
        triplet,
        detector_pixels=(256, 256),
    )

print("Compound components:", [c.name for c in scene_triplet.component_registry.compounds])
print("Sources            :", list(scene_triplet.source_registry._registry.keys()))
print("Detectors          :", list(scene_triplet.detector_registry._registry.keys()))
print(f"Total surfaces     : {len(scene_triplet.surfaces)}")

Compound components: ['L1', 'L3', 'L5']
Sources            : ['S0', 'S1', 'S2']
Detectors          : ['D1']
Total surfaces     : 12


## 2. Baseline Trace

For a Cooke Triplet the direct optical path crosses 6 refractive surfaces
(two per lens). The `max_depth` parameter counts every surface hit, so the
minimum value that allows rays through the full system is roughly equal to the
number of surfaces in the scene. Here we use `max_depth=200`
for a full baseline image with all Fresnel reflections included.

In [3]:
result_base = scene_triplet.trace(num_rays=100_000, max_depth=200, seed=42)
irr_base = result_base.detectors['D1']

print(f"Baseline flux on detector: {irr_base.total_flux:.5f} W")
print(f"Rays on detector         : {irr_base.num_rays_hit:,}")

fig = irr_base.plot(cmap='hot')
plt.title('Cooke Triplet — baseline trace (max_depth=200)')
plt.tight_layout()
plt.show()
plt.close(fig)

C:\Users\kdani\Documents\Python_Scripts\optiland\optiland\nonsequential\components\refractive.py:88: RuntimeWarning: invalid value encountered in multiply
  rays.x = be.where(hit_mask, rays.x + t * rays.L, rays.x)


Baseline flux on detector: 1.69266 W
Rays on detector         : 56,422


C:\Users\kdani\AppData\Local\Temp\ipykernel_27788\3489207307.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Limited-Bounce Comparison

Lowering `max_depth` restricts how many surface interactions each ray can
have. Rays exceeding the limit are killed without reaching the detector.
Here we compare a tight limit (few ghost paths possible) against the full trace:

In [4]:
# Rebuild scene each trace (detectors accumulate across calls)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    scene_ghosts = sequential_to_nonsequential(
        CookeTriplet(),
        detector_pixels=(256, 256),
    )

result_ghosts = scene_ghosts.trace(
    num_rays=200_000,
    max_depth=10,
    min_flux_fraction=1e-8,  # track very faint rays
    seed=42,
)
irr_ghosts = result_ghosts.detectors['D1']

print(f"Multi-bounce flux on detector : {irr_ghosts.total_flux:.5f} W")
print(f"Rays on detector              : {irr_ghosts.num_rays_hit:,}")

fig = irr_ghosts.plot(cmap='hot')
plt.title('Multi-bounce (max_depth=10) — limited ghost reflections')
plt.tight_layout()
plt.show()
plt.close(fig)

C:\Users\kdani\Documents\Python_Scripts\optiland\optiland\nonsequential\components\refractive.py:88: RuntimeWarning: invalid value encountered in multiply
  rays.x = be.where(hit_mask, rays.x + t * rays.L, rays.x)


Multi-bounce flux on detector : 1.68175 W
Rays on detector              : 112,117


C:\Users\kdani\AppData\Local\Temp\ipykernel_27788\4094342172.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Ghost Contribution as a Function of Bounces

Track how the detected flux changes as we allow more reflections:

In [5]:
bounce_levels = [10, 20, 50, 100, 200]
detected_fluxes = []

for max_b in bounce_levels:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        sc = sequential_to_nonsequential(CookeTriplet(), detector_pixels=(128, 128))
    r = sc.trace(num_rays=50_000, max_depth=max_b, min_flux_fraction=1e-8, seed=42)
    detected_fluxes.append(r.detectors['D1'].total_flux)

# Compute ghost fraction relative to lowest-bounce baseline (guard against zero)
base = detected_fluxes[0] if detected_fluxes[0] > 1e-12 else max(detected_fluxes)
ghost_fractions = [
    (f - base) / base * 100
    for f in detected_fluxes
]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(bounce_levels, detected_fluxes, 'o-')
axes[0].set_xlabel('max_depth')
axes[0].set_ylabel('Flux on detector [W]')
axes[0].set_title('Detected flux vs. bounce limit')
axes[0].grid(True, alpha=0.4)

axes[1].plot(bounce_levels, ghost_fractions, 's-', color='orange')
axes[1].set_xlabel('max_depth')
axes[1].set_ylabel('Flux change vs. baseline [%]')
axes[1].set_title('Ghost/stray contribution vs. bounce limit')
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()
plt.close(fig)

for b, f, g in zip(bounce_levels, detected_fluxes, ghost_fractions):
    print(f"max_depth={b:>3}: flux={f:.5f} W  delta={g:+.3f}%")

C:\Users\kdani\Documents\Python_Scripts\optiland\optiland\nonsequential\components\refractive.py:88: RuntimeWarning: invalid value encountered in multiply
  rays.x = be.where(hit_mask, rays.x + t * rays.L, rays.x)


C:\Users\kdani\Documents\Python_Scripts\optiland\optiland\nonsequential\components\refractive.py:88: RuntimeWarning: invalid value encountered in multiply
  rays.x = be.where(hit_mask, rays.x + t * rays.L, rays.x)


C:\Users\kdani\Documents\Python_Scripts\optiland\optiland\nonsequential\components\refractive.py:88: RuntimeWarning: invalid value encountered in multiply
  rays.x = be.where(hit_mask, rays.x + t * rays.L, rays.x)


C:\Users\kdani\Documents\Python_Scripts\optiland\optiland\nonsequential\components\refractive.py:88: RuntimeWarning: invalid value encountered in multiply
  rays.x = be.where(hit_mask, rays.x + t * rays.L, rays.x)


C:\Users\kdani\Documents\Python_Scripts\optiland\optiland\nonsequential\components\refractive.py:88: RuntimeWarning: invalid value encountered in multiply
  rays.x = be.where(hit_mask, rays.x + t * rays.L, rays.x)


max_depth= 10: flux=1.68359 W  delta=+0.000%
max_depth= 20: flux=1.69043 W  delta=+0.406%
max_depth= 50: flux=1.68893 W  delta=+0.317%
max_depth=100: flux=1.68893 W  delta=+0.317%
max_depth=200: flux=1.68893 W  delta=+0.317%


C:\Users\kdani\AppData\Local\Temp\ipykernel_27788\2730448150.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Manual NSQ Stray-Light Scene

You can also build an NSQ scene from scratch and use a large `max_depth`
for stray-light studies on custom systems:

In [6]:
spec = Spectrum.monochromatic(0.55)

# Two-lens system: investigate stray light from inter-lens reflections
scene_stray = NSQScene()
scene_stray.add_source(
    'S', CoordinateSystem(z=-80),
    CollimatedSourceConfig(spectrum=spec, total_flux=1.0, aperture_radius=10.0),
)
scene_stray.add_lens(
    'L1', CoordinateSystem(z=0),
    LensConfig(r1=50, r2=-50, thickness=5, material='N-BK7', front_aperture_radius=12.5),
)
scene_stray.add_lens(
    'L2', CoordinateSystem(z=40),
    LensConfig(r1=-80, r2=80, thickness=4, material='N-SF5', front_aperture_radius=12.5),
)
scene_stray.add_detector(
    'D', CoordinateSystem(z=120),
    IrradianceDetectorConfig(width=20, height=20, num_pixels_x=128, num_pixels_y=128),
)

# Low bounce: direct path only
r_low  = scene_stray.trace(num_rays=50_000, max_depth=2,  seed=42)
# Higher bounce: includes inter-element ghosts
r_high = scene_stray.trace(num_rays=50_000, max_depth=10, seed=42)

i_low  = r_low.detectors['D']
i_high = r_high.detectors['D']
print(f"max_depth=2  : {i_low.total_flux:.5f} W  ({i_low.num_rays_hit:,} rays)")
print(f"max_depth=10 : {i_high.total_flux:.5f} W  ({i_high.num_rays_hit:,} rays)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, irr, title in zip(axes, [i_low, i_high],
                          ['max_depth=2', 'max_depth=10']):
    im = ax.imshow(irr.irradiance, origin='lower', cmap='hot', aspect='equal',
                   extent=[irr.x_coords[0], irr.x_coords[-1],
                            irr.y_coords[0], irr.y_coords[-1]])
    plt.colorbar(im, ax=ax, label='W/mm²')
    ax.set_title(title)
    ax.set_xlabel('x [mm]'); ax.set_ylabel('y [mm]')
plt.tight_layout()
plt.show()
plt.close(fig)

max_depth=2  : 0.00000 W  (0 rays)
max_depth=10 : 0.66990 W  (33,495 rays)


C:\Users\kdani\AppData\Local\Temp\ipykernel_27788\2430726958.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

- `sequential_to_nonsequential(optic)` converts an `Optic` to an `NSQScene` automatically
- The NSQ tracer applies **Fresnel splitting** at every uncoated interface
- `max_depth=1` → single-pass (no ghosts); increase to reveal stray light
- `min_flux_fraction` controls when very faint ghost rays are killed
- Track `result.detectors['D1'].total_flux` across bounce levels to quantify ghost contributions
- Ghost reflections are suppressed by AR coatings — add `SurfaceConfig(bsdf=...)` after conversion